In [1]:
# Clone your GitHub repo (you’ll be prompted to authorize if it's private)
!git clone https://github.com/colterwood/LHL-final-final-project.git

Cloning into 'LHL-final-final-project'...
remote: Enumerating objects: 320, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 320 (delta 37), reused 55 (delta 14), pack-reused 224 (from 1)
Receiving objects: 100% (320/320), 5.39 MiB | 3.76 MiB/s, done.
Resolving deltas: 100% (152/152), done.


In [7]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup, Comment
import requests
from io import StringIO
import string
import time
import re
from functools import reduce
import os

In [31]:
year = 2024
url = f"https://www.basketball-reference.com/wnba/years/{year}.html"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, "html.parser")
comments = soup.find_all(string=lambda text: isinstance(text, Comment))

In [32]:
# full list of table IDs to load from the team page
table_ids = [
    "per_game-team", "per_game-opponent",
    "totals-team", "totals-opponent",
    "advanced-team", "per_poss-team", "per_poss-opponent",
    "shooting-team", "shooting-opponent"
]

# columns to exclude from prefixing
no_prefix_cols = {"Team", "G", "MP"}

# dictionary to store DataFrames by table ID
team_tables = {}

In [33]:
def load_table(table_id, soup, comments):
    # try to find visible table
    tag = soup.find("table", {"id": table_id})

    # if not visible, look in comments
    if tag is None:
        for c in comments:
            if f'id="{table_id}"' in c:
                tag = BeautifulSoup(c, "html.parser").find("table", {"id": table_id})
                break
    if tag is None:
        print(f"Table not found: {table_id}")
        return None

    # shooting tables use multilevel headers
    if "shooting" in table_id:
        df = pd.read_html(StringIO(str(tag)), header=[0, 1])[0]
        df.columns = [f"{a}_{b}" if not a.startswith("Unnamed") else b for a, b in df.columns]
    else:
        df = pd.read_html(StringIO(str(tag)), header=0)[0]

    # prefix columns except base ones
    prefix = table_id.split("-")[0]
    df.columns = [col if col in no_prefix_cols else f"{prefix}_{col}" for col in df.columns]
    return df

In [34]:
team_tables = []

for table_id in table_ids:
    print(f"Loading: {table_id}")
    df = load_table(table_id, soup, comments)
    if df is not None:
        team_tables.append((table_id, df))

Loading: per_game-team
Loading: per_game-opponent
Loading: totals-team
Loading: totals-opponent
Loading: advanced-team
Loading: per_poss-team
Loading: per_poss-opponent
Loading: shooting-team
Loading: shooting-opponent


In [36]:
for name, df in team_tables:
    print(f"{name}: {df.shape}")

per_game-team: (13, 25)
per_game-opponent: (13, 25)
totals-team: (13, 25)
totals-opponent: (13, 25)
advanced-team: (14, 29)
per_poss-team: (12, 25)
per_poss-opponent: (12, 25)
shooting-team: (13, 26)
shooting-opponent: (13, 26)


In [39]:
for name, df in team_tables:
    if name == "per_game-team":
        print(name, df.shape)
        display(df.head())
        break

per_game-team (13, 24)


,Team,G,MP,per_game_FG,per_game_FGA,per_game_FG%,per_game_3P,per_game_3PA,per_game_3P%,per_game_2P,...,per_game_FT%,per_game_ORB,per_game_DRB,per_game_TRB,per_game_AST,per_game_STL,per_game_BLK,per_game_TOV,per_game_PF,per_game_PTS
0,Las Vegas Aces*,40,200.6,30.9,68.1,0.454,9.4,26.5,0.355,21.5,...,0.828,5.6,28.5,34.1,20.5,7.1,5.0,10.8,16.5,86.4
1,New York Liberty*,40,200.0,30.8,68.7,0.448,10.1,29.0,0.349,20.6,...,0.814,8.5,28.1,36.6,22.8,7.9,4.5,12.7,15.4,85.6
2,Indiana Fever*,40,200.6,31.3,68.5,0.456,9.2,25.9,0.356,22.1,...,0.775,8.3,26.8,35.1,20.4,5.9,4.3,14.2,18.2,85.0
3,Dallas Wings,40,201.9,31.7,71.0,0.446,6.3,19.2,0.326,25.4,...,0.786,10.5,24.3,34.8,20.4,7.1,4.0,14.8,18.5,84.2
4,Seattle Storm*,40,201.2,31.1,71.3,0.435,6.1,21.0,0.288,25.0,...,0.840,8.7,26.0,34.7,20.7,9.3,5.2,12.4,16.5,83.2


In [40]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["per_game_Rk"]) if name == "per_game-team" and "per_game_Rk" in df.columns else df)
               for name, df in team_tables]

In [41]:
for name, df in team_tables:
    if name == "per_game-team":
        print(name, df.shape)
        display(df.head())
        break

per_game-team (13, 24)


,Team,G,MP,per_game_FG,per_game_FGA,per_game_FG%,per_game_3P,per_game_3PA,per_game_3P%,per_game_2P,...,per_game_FT%,per_game_ORB,per_game_DRB,per_game_TRB,per_game_AST,per_game_STL,per_game_BLK,per_game_TOV,per_game_PF,per_game_PTS
0,Las Vegas Aces*,40,200.6,30.9,68.1,0.454,9.4,26.5,0.355,21.5,...,0.828,5.6,28.5,34.1,20.5,7.1,5.0,10.8,16.5,86.4
1,New York Liberty*,40,200.0,30.8,68.7,0.448,10.1,29.0,0.349,20.6,...,0.814,8.5,28.1,36.6,22.8,7.9,4.5,12.7,15.4,85.6
2,Indiana Fever*,40,200.6,31.3,68.5,0.456,9.2,25.9,0.356,22.1,...,0.775,8.3,26.8,35.1,20.4,5.9,4.3,14.2,18.2,85.0
3,Dallas Wings,40,201.9,31.7,71.0,0.446,6.3,19.2,0.326,25.4,...,0.786,10.5,24.3,34.8,20.4,7.1,4.0,14.8,18.5,84.2
4,Seattle Storm*,40,201.2,31.1,71.3,0.435,6.1,21.0,0.288,25.0,...,0.840,8.7,26.0,34.7,20.7,9.3,5.2,12.4,16.5,83.2


In [42]:
for name, df in team_tables:
    if name == "per_game-opponent":
        print(name, df.shape)
        display(df.head())
        break

per_game-opponent (13, 25)


,per_game_Rk,Team,G,MP,per_game_FG,per_game_FGA,per_game_FG%,per_game_3P,per_game_3PA,per_game_3P%,...,per_game_FT%,per_game_ORB,per_game_DRB,per_game_TRB,per_game_AST,per_game_STL,per_game_BLK,per_game_TOV,per_game_PF,per_game_PTS
0,1.0,Connecticut Sun*,40,201.2,27.2,63.0,0.431,6.5,20.6,0.313,...,0.779,7.0,24.7,31.7,19.1,7.0,4.6,15.0,18.1,73.6
1,2.0,Minnesota Lynx*,40,201.9,28.2,68.7,0.410,6.8,22.7,0.301,...,0.788,9.4,25.9,35.3,18.6,7.5,3.7,14.9,15.7,75.6
2,3.0,New York Liberty*,40,200.0,29.1,68.3,0.425,7.0,21.5,0.324,...,0.769,7.4,25.3,32.7,19.4,6.2,3.2,12.7,16.9,76.5
3,4.0,Seattle Storm*,40,201.2,28.8,67.7,0.426,6.9,21.0,0.329,...,0.784,8.9,27.2,36.0,19.5,7.6,4.6,15.0,16.4,78.8
4,5.0,Atlanta Dream*,40,201.9,28.9,67.4,0.429,8.0,23.1,0.344,...,0.797,7.5,27.2,34.6,20.1,6.9,4.1,12.5,17.6,79.8


In [43]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["per_game_Rk"]) if name == "per_game-opponent" and "per_game_Rk" in df.columns else df)
               for name, df in team_tables]

In [44]:
for name, df in team_tables:
    if name == "per_game-opponent":
        print(name, df.shape)
        display(df.head())
        break

per_game-opponent (13, 24)


,Team,G,MP,per_game_FG,per_game_FGA,per_game_FG%,per_game_3P,per_game_3PA,per_game_3P%,per_game_2P,...,per_game_FT%,per_game_ORB,per_game_DRB,per_game_TRB,per_game_AST,per_game_STL,per_game_BLK,per_game_TOV,per_game_PF,per_game_PTS
0,Connecticut Sun*,40,201.2,27.2,63.0,0.431,6.5,20.6,0.313,20.7,...,0.779,7.0,24.7,31.7,19.1,7.0,4.6,15.0,18.1,73.6
1,Minnesota Lynx*,40,201.9,28.2,68.7,0.410,6.8,22.7,0.301,21.4,...,0.788,9.4,25.9,35.3,18.6,7.5,3.7,14.9,15.7,75.6
2,New York Liberty*,40,200.0,29.1,68.3,0.425,7.0,21.5,0.324,22.1,...,0.769,7.4,25.3,32.7,19.4,6.2,3.2,12.7,16.9,76.5
3,Seattle Storm*,40,201.2,28.8,67.7,0.426,6.9,21.0,0.329,21.9,...,0.784,8.9,27.2,36.0,19.5,7.6,4.6,15.0,16.4,78.8
4,Atlanta Dream*,40,201.9,28.9,67.4,0.429,8.0,23.1,0.344,21.0,...,0.797,7.5,27.2,34.6,20.1,6.9,4.1,12.5,17.6,79.8


In [45]:
for name, df in team_tables:
    if name == "totals-team":
        print(name, df.shape)
        display(df.head())
        break

totals-team (13, 25)


,totals_Rk,Team,G,MP,totals_FG,totals_FGA,totals_FG%,totals_3P,totals_3PA,totals_3P%,...,totals_FT%,totals_ORB,totals_DRB,totals_TRB,totals_AST,totals_STL,totals_BLK,totals_TOV,totals_PF,totals_PTS
0,1.0,Las Vegas Aces*,40,8024,1236,2725,0.454,376,1058,0.355,...,0.828,223,1141,1364,820,282,198,432,661,3455
1,2.0,New York Liberty*,40,7999,1230,2747,0.448,405,1160,0.349,...,0.814,338,1125,1463,911,316,179,506,615,3424
2,3.0,Indiana Fever*,40,8024,1250,2741,0.456,368,1034,0.356,...,0.775,331,1071,1402,816,235,173,568,726,3399
3,4.0,Dallas Wings,40,8074,1266,2841,0.446,250,767,0.326,...,0.786,419,971,1390,817,285,158,593,739,3368
4,5.0,Seattle Storm*,40,8049,1242,2852,0.435,242,840,0.288,...,0.840,346,1040,1386,826,372,206,496,660,3329


In [48]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["totals_Rk"]) if name == "totals-team" and "totals_Rk" in df.columns else df)
               for name, df in team_tables]

In [49]:
for name, df in team_tables:
    if name == "totals-team":
        print(name, df.shape)
        display(df.head())
        break

totals-team (13, 24)


,Team,G,MP,totals_FG,totals_FGA,totals_FG%,totals_3P,totals_3PA,totals_3P%,totals_2P,...,totals_FT%,totals_ORB,totals_DRB,totals_TRB,totals_AST,totals_STL,totals_BLK,totals_TOV,totals_PF,totals_PTS
0,Las Vegas Aces*,40,8024,1236,2725,0.454,376,1058,0.355,860,...,0.828,223,1141,1364,820,282,198,432,661,3455
1,New York Liberty*,40,7999,1230,2747,0.448,405,1160,0.349,825,...,0.814,338,1125,1463,911,316,179,506,615,3424
2,Indiana Fever*,40,8024,1250,2741,0.456,368,1034,0.356,882,...,0.775,331,1071,1402,816,235,173,568,726,3399
3,Dallas Wings,40,8074,1266,2841,0.446,250,767,0.326,1016,...,0.786,419,971,1390,817,285,158,593,739,3368
4,Seattle Storm*,40,8049,1242,2852,0.435,242,840,0.288,1000,...,0.840,346,1040,1386,826,372,206,496,660,3329


In [50]:
for name, df in team_tables:
    if name == "totals-opponent":
        print(name, df.shape)
        display(df.head())
        break

totals-opponent (13, 25)


,totals_Rk,Team,G,MP,totals_FG,totals_FGA,totals_FG%,totals_3P,totals_3PA,totals_3P%,...,totals_FT%,totals_ORB,totals_DRB,totals_TRB,totals_AST,totals_STL,totals_BLK,totals_TOV,totals_PF,totals_PTS
0,1.0,Connecticut Sun*,40,8049,1086.0,2520.0,0.431,258.0,825.0,0.313,...,0.779,280.0,989.0,1269.0,764.0,280.0,183.0,599.0,725.0,2944.0
1,2.0,Minnesota Lynx*,40,8074,1127.0,2749.0,0.410,273.0,907.0,0.301,...,0.788,374.0,1037.0,1411.0,742.0,298.0,146.0,596.0,626.0,3024.0
2,3.0,New York Liberty*,40,7999,1162.0,2732.0,0.425,279.0,860.0,0.324,...,0.769,296.0,1010.0,1306.0,777.0,249.0,127.0,506.0,674.0,3058.0
3,4.0,Seattle Storm*,40,8049,1152.0,2707.0,0.426,277.0,841.0,0.329,...,0.784,355.0,1086.0,1441.0,779.0,305.0,183.0,599.0,656.0,3150.0
4,5.0,Atlanta Dream*,40,8074,1156.0,2695.0,0.429,318.0,924.0,0.344,...,0.797,298.0,1086.0,1384.0,804.0,275.0,164.0,501.0,702.0,3190.0


In [51]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["totals_Rk"]) if name == "totals-opponent" and "totals_Rk" in df.columns else df)
               for name, df in team_tables]

In [52]:
for name, df in team_tables:
    if name == "totals-opponent":
        print(name, df.shape)
        display(df.head())
        break

totals-opponent (13, 24)


,Team,G,MP,totals_FG,totals_FGA,totals_FG%,totals_3P,totals_3PA,totals_3P%,totals_2P,...,totals_FT%,totals_ORB,totals_DRB,totals_TRB,totals_AST,totals_STL,totals_BLK,totals_TOV,totals_PF,totals_PTS
0,Connecticut Sun*,40,8049,1086.0,2520.0,0.431,258.0,825.0,0.313,828.0,...,0.779,280.0,989.0,1269.0,764.0,280.0,183.0,599.0,725.0,2944.0
1,Minnesota Lynx*,40,8074,1127.0,2749.0,0.410,273.0,907.0,0.301,854.0,...,0.788,374.0,1037.0,1411.0,742.0,298.0,146.0,596.0,626.0,3024.0
2,New York Liberty*,40,7999,1162.0,2732.0,0.425,279.0,860.0,0.324,883.0,...,0.769,296.0,1010.0,1306.0,777.0,249.0,127.0,506.0,674.0,3058.0
3,Seattle Storm*,40,8049,1152.0,2707.0,0.426,277.0,841.0,0.329,875.0,...,0.784,355.0,1086.0,1441.0,779.0,305.0,183.0,599.0,656.0,3150.0
4,Atlanta Dream*,40,8074,1156.0,2695.0,0.429,318.0,924.0,0.344,838.0,...,0.797,298.0,1086.0,1384.0,804.0,275.0,164.0,501.0,702.0,3190.0


In [54]:
for name, df in team_tables:
    if name == "per_poss-team":
        print(name, df.shape)
        display(df.head())
        break

per_poss-team (12, 25)


,per_poss_Rk,Team,G,MP,per_poss_FG,per_poss_FGA,per_poss_FG%,per_poss_3P,per_poss_3PA,per_poss_3P%,...,per_poss_FT%,per_poss_ORB,per_poss_DRB,per_poss_TRB,per_poss_AST,per_poss_STL,per_poss_BLK,per_poss_TOV,per_poss_PF,per_poss_PTS
0,1,New York Liberty*,40,7999,39.4,88.0,0.448,13.0,37.1,0.349,...,0.814,10.8,36.0,46.8,29.2,10.1,5.7,16.2,19.7,109.6
1,2,Las Vegas Aces*,40,8024,38.6,85.2,0.454,11.8,33.1,0.355,...,0.828,7.0,35.7,42.6,25.6,8.8,6.2,13.5,20.7,108.0
2,3,Indiana Fever*,40,8024,39.0,85.6,0.456,11.5,32.3,0.356,...,0.775,10.3,33.4,43.8,25.5,7.3,5.4,17.7,22.7,106.1
3,4,Connecticut Sun*,40,8049,38.3,86.4,0.444,7.7,23.6,0.327,...,0.753,10.9,32.9,43.8,26.1,10.7,4.8,15.8,21.1,105.0
4,5,Minnesota Lynx*,40,8074,38.4,85.9,0.448,12.1,31.9,0.380,...,0.790,9.5,34.2,43.7,29.4,10.9,5.4,17.1,20.9,104.6


In [55]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["per_poss_Rk"]) if name == "per_poss-team" and "per_poss_Rk" in df.columns else df)
               for name, df in team_tables]

In [56]:
for name, df in team_tables:
    if name == "per_poss-team":
        print(name, df.shape)
        display(df.head())
        break

per_poss-team (12, 24)


,Team,G,MP,per_poss_FG,per_poss_FGA,per_poss_FG%,per_poss_3P,per_poss_3PA,per_poss_3P%,per_poss_2P,...,per_poss_FT%,per_poss_ORB,per_poss_DRB,per_poss_TRB,per_poss_AST,per_poss_STL,per_poss_BLK,per_poss_TOV,per_poss_PF,per_poss_PTS
0,New York Liberty*,40,7999,39.4,88.0,0.448,13.0,37.1,0.349,26.4,...,0.814,10.8,36.0,46.8,29.2,10.1,5.7,16.2,19.7,109.6
1,Las Vegas Aces*,40,8024,38.6,85.2,0.454,11.8,33.1,0.355,26.9,...,0.828,7.0,35.7,42.6,25.6,8.8,6.2,13.5,20.7,108.0
2,Indiana Fever*,40,8024,39.0,85.6,0.456,11.5,32.3,0.356,27.5,...,0.775,10.3,33.4,43.8,25.5,7.3,5.4,17.7,22.7,106.1
3,Connecticut Sun*,40,8049,38.3,86.4,0.444,7.7,23.6,0.327,30.6,...,0.753,10.9,32.9,43.8,26.1,10.7,4.8,15.8,21.1,105.0
4,Minnesota Lynx*,40,8074,38.4,85.9,0.448,12.1,31.9,0.380,26.3,...,0.790,9.5,34.2,43.7,29.4,10.9,5.4,17.1,20.9,104.6


In [57]:
for name, df in team_tables:
    if name == "per_poss-opponent":
        print(name, df.shape)
        display(df.head())
        break

per_poss-opponent (12, 25)


,per_poss_Rk,Team,G,MP,per_poss_FG,per_poss_FGA,per_poss_FG%,per_poss_3P,per_poss_3PA,per_poss_3P%,...,per_poss_FT%,per_poss_ORB,per_poss_DRB,per_poss_TRB,per_poss_AST,per_poss_STL,per_poss_BLK,per_poss_TOV,per_poss_PF,per_poss_PTS
0,1,Connecticut Sun*,40,8049,35.6,82.6,0.431,8.5,27.0,0.313,...,0.779,9.2,32.4,41.6,25.0,9.2,6.0,19.6,23.8,96.4
1,2,Minnesota Lynx*,40,8074,36.0,87.7,0.410,8.7,28.9,0.301,...,0.788,11.9,33.1,45.0,23.7,9.5,4.7,19.0,20.0,96.5
2,3,New York Liberty*,40,7999,37.2,87.5,0.425,8.9,27.5,0.324,...,0.769,9.5,32.3,41.8,24.9,8.0,4.1,16.2,21.6,97.9
3,4,Seattle Storm*,40,8049,36.0,84.7,0.426,8.7,26.3,0.329,...,0.784,11.1,34.0,45.1,24.4,9.5,5.7,18.7,20.5,98.6
4,5,Las Vegas Aces*,40,8024,37.6,86.9,0.433,9.6,27.5,0.350,...,0.761,9.2,35.1,44.3,24.9,8.1,4.5,15.8,21.5,101.2


In [58]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["per_poss_Rk"]) if name == "per_poss-opponent" and "per_poss_Rk" in df.columns else df)
               for name, df in team_tables]

In [59]:
for name, df in team_tables:
    if name == "per_poss-opponent":
        print(name, df.shape)
        display(df.head())
        break

per_poss-opponent (12, 24)


,Team,G,MP,per_poss_FG,per_poss_FGA,per_poss_FG%,per_poss_3P,per_poss_3PA,per_poss_3P%,per_poss_2P,...,per_poss_FT%,per_poss_ORB,per_poss_DRB,per_poss_TRB,per_poss_AST,per_poss_STL,per_poss_BLK,per_poss_TOV,per_poss_PF,per_poss_PTS
0,Connecticut Sun*,40,8049,35.6,82.6,0.431,8.5,27.0,0.313,27.1,...,0.779,9.2,32.4,41.6,25.0,9.2,6.0,19.6,23.8,96.4
1,Minnesota Lynx*,40,8074,36.0,87.7,0.410,8.7,28.9,0.301,27.2,...,0.788,11.9,33.1,45.0,23.7,9.5,4.7,19.0,20.0,96.5
2,New York Liberty*,40,7999,37.2,87.5,0.425,8.9,27.5,0.324,28.3,...,0.769,9.5,32.3,41.8,24.9,8.0,4.1,16.2,21.6,97.9
3,Seattle Storm*,40,8049,36.0,84.7,0.426,8.7,26.3,0.329,27.4,...,0.784,11.1,34.0,45.1,24.4,9.5,5.7,18.7,20.5,98.6
4,Las Vegas Aces*,40,8024,37.6,86.9,0.433,9.6,27.5,0.350,27.9,...,0.761,9.2,35.1,44.3,24.9,8.1,4.5,15.8,21.5,101.2


In [60]:
for name, df in team_tables:
    if name == "shooting-team":
        print(name, df.shape)
        display(df.head())
        break

shooting-team (13, 26)


,shooting_Rk,Team,G,MP,shooting_FG%,shooting_Dist.,shooting_Unnamed: 6_level_1,shooting_% of FGA by Distance_2P,shooting_% of FGA by Distance_0-3,shooting_% of FGA by Distance_3-10,...,shooting_FG% by Distance_3-10,shooting_FG% by Distance_10-16,shooting_FG% by Distance_16-3P,shooting_FG% by Distance_3P,shooting_Unnamed: 20_level_1,shooting_% of FG Ast'd_2P,shooting_% of FG Ast'd_3P,shooting_Unnamed: 23_level_1,shooting_Corner_%3PA,shooting_Corner_3P%
0,1.0,Atlanta Dream*,40,8074,0.408,12.8,NaN,0.715,0.228,0.226,...,0.374,0.339,0.395,0.308,NaN,0.599,0.891,NaN,0.210,0.288
1,2.0,Chicago Sky,40,8000,0.422,10.8,NaN,0.788,0.298,0.262,...,0.377,0.358,0.344,0.324,NaN,0.606,0.808,NaN,0.195,0.310
2,3.0,Connecticut Sun*,40,8049,0.444,11.9,NaN,0.727,0.267,0.262,...,0.416,0.414,0.345,0.327,NaN,0.635,0.860,NaN,0.181,0.300
3,4.0,Dallas Wings,40,8074,0.446,12.3,NaN,0.730,0.242,0.250,...,0.443,0.337,0.383,0.326,NaN,0.614,0.772,NaN,0.149,0.333
4,5.0,Indiana Fever*,40,8024,0.456,13.6,NaN,0.623,0.280,0.189,...,0.478,0.375,0.372,0.356,NaN,0.602,0.774,NaN,0.182,0.372


In [61]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["shooting_Rk"]) if name == "shooting-team" and "shooting_Rk" in df.columns else df)
               for name, df in team_tables]

In [62]:
for name, df in team_tables:
    if name == "shooting-team":
        print(name, df.shape)
        display(df.head())
        break

shooting-team (13, 25)


,Team,G,MP,shooting_FG%,shooting_Dist.,shooting_Unnamed: 6_level_1,shooting_% of FGA by Distance_2P,shooting_% of FGA by Distance_0-3,shooting_% of FGA by Distance_3-10,shooting_% of FGA by Distance_10-16,...,shooting_FG% by Distance_3-10,shooting_FG% by Distance_10-16,shooting_FG% by Distance_16-3P,shooting_FG% by Distance_3P,shooting_Unnamed: 20_level_1,shooting_% of FG Ast'd_2P,shooting_% of FG Ast'd_3P,shooting_Unnamed: 23_level_1,shooting_Corner_%3PA,shooting_Corner_3P%
0,Atlanta Dream*,40,8074,0.408,12.8,NaN,0.715,0.228,0.226,0.137,...,0.374,0.339,0.395,0.308,NaN,0.599,0.891,NaN,0.210,0.288
1,Chicago Sky,40,8000,0.422,10.8,NaN,0.788,0.298,0.262,0.120,...,0.377,0.358,0.344,0.324,NaN,0.606,0.808,NaN,0.195,0.310
2,Connecticut Sun*,40,8049,0.444,11.9,NaN,0.727,0.267,0.262,0.110,...,0.416,0.414,0.345,0.327,NaN,0.635,0.860,NaN,0.181,0.300
3,Dallas Wings,40,8074,0.446,12.3,NaN,0.730,0.242,0.250,0.130,...,0.443,0.337,0.383,0.326,NaN,0.614,0.772,NaN,0.149,0.333
4,Indiana Fever*,40,8024,0.456,13.6,NaN,0.623,0.280,0.189,0.073,...,0.478,0.375,0.372,0.356,NaN,0.602,0.774,NaN,0.182,0.372


In [63]:
for name, df in team_tables:
    if name == "shooting-opponent":
        print(name, df.shape)
        display(df.head())
        break

shooting-opponent (13, 26)


,shooting_Rk,Team,G,MP,shooting_FG%,shooting_Dist.,shooting_Unnamed: 6_level_1,shooting_% of FGA by Distance_2P,shooting_% of FGA by Distance_0-3,shooting_% of FGA by Distance_3-10,...,shooting_FG% by Distance_3-10,shooting_FG% by Distance_10-16,shooting_FG% by Distance_16-3P,shooting_FG% by Distance_3P,shooting_Unnamed: 20_level_1,shooting_% of FG Ast'd_2P,shooting_% of FG Ast'd_3P,shooting_Unnamed: 23_level_1,shooting_Corner_%3PA,shooting_Corner_3P%
0,1.0,Atlanta Dream*,40,8075,0.429,13.0,NaN,0.657,0.243,0.223,...,0.400,0.358,0.397,0.344,NaN,0.628,0.874,NaN,0.215,0.357
1,2.0,Chicago Sky,40,8000,0.446,13.0,NaN,0.678,0.275,0.200,...,0.391,0.393,0.433,0.326,NaN,0.672,0.880,NaN,0.203,0.278
2,3.0,Connecticut Sun*,40,8050,0.431,13.0,NaN,0.673,0.207,0.257,...,0.435,0.401,0.362,0.313,NaN,0.644,0.895,NaN,0.193,0.346
3,4.0,Dallas Wings,40,8075,0.475,13.0,NaN,0.668,0.232,0.236,...,0.468,0.400,0.397,0.365,NaN,0.620,0.898,NaN,0.197,0.357
4,5.0,Indiana Fever*,40,8025,0.441,14.0,NaN,0.638,0.228,0.209,...,0.448,0.375,0.408,0.361,NaN,0.603,0.889,NaN,0.193,0.396


In [64]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["shooting_Rk"]) if name == "shooting-opponent" and "shooting_Rk" in df.columns else df)
               for name, df in team_tables]

In [65]:
for name, df in team_tables:
    if name == "shooting-opponent":
        print(name, df.shape)
        display(df.head())
        break

shooting-opponent (13, 25)


,Team,G,MP,shooting_FG%,shooting_Dist.,shooting_Unnamed: 6_level_1,shooting_% of FGA by Distance_2P,shooting_% of FGA by Distance_0-3,shooting_% of FGA by Distance_3-10,shooting_% of FGA by Distance_10-16,...,shooting_FG% by Distance_3-10,shooting_FG% by Distance_10-16,shooting_FG% by Distance_16-3P,shooting_FG% by Distance_3P,shooting_Unnamed: 20_level_1,shooting_% of FG Ast'd_2P,shooting_% of FG Ast'd_3P,shooting_Unnamed: 23_level_1,shooting_Corner_%3PA,shooting_Corner_3P%
0,Atlanta Dream*,40,8075,0.429,13.0,NaN,0.657,0.243,0.223,0.102,...,0.400,0.358,0.397,0.344,NaN,0.628,0.874,NaN,0.215,0.357
1,Chicago Sky,40,8000,0.446,13.0,NaN,0.678,0.275,0.200,0.100,...,0.391,0.393,0.433,0.326,NaN,0.672,0.880,NaN,0.203,0.278
2,Connecticut Sun*,40,8050,0.431,13.0,NaN,0.673,0.207,0.257,0.120,...,0.435,0.401,0.362,0.313,NaN,0.644,0.895,NaN,0.193,0.346
3,Dallas Wings,40,8075,0.475,13.0,NaN,0.668,0.232,0.236,0.106,...,0.468,0.400,0.397,0.365,NaN,0.620,0.898,NaN,0.197,0.357
4,Indiana Fever*,40,8025,0.441,14.0,NaN,0.638,0.228,0.209,0.112,...,0.448,0.375,0.408,0.361,NaN,0.603,0.889,NaN,0.193,0.396


In [66]:
for name, df in team_tables:
    if name == "advanced-team":
        print(name, df.shape)
        display(df.head())
        break

advanced-team (14, 29)


,advanced_Unnamed: 0,advanced_Unnamed: 1,advanced_Unnamed: 2,advanced_Unnamed: 3,advanced_Unnamed: 4,advanced_Unnamed: 5,advanced_Unnamed: 6,advanced_Unnamed: 7,advanced_Unnamed: 8,advanced_Unnamed: 9,...,advanced_Offense Four Factors.1,advanced_Offense Four Factors.2,advanced_Offense Four Factors.3,advanced_Unnamed: 22,advanced_Defense Four Factors,advanced_Defense Four Factors.1,advanced_Defense Four Factors.2,advanced_Defense Four Factors.3,advanced_Unnamed: 27,advanced_Unnamed: 28
0,Rk,Team,Age,W,L,PW,PL,MOV,SOS,SRS,...,TOV%,ORB%,FT/FGA,NaN,eFG%,TOV%,DRB%,FT/FGA,NaN,Arena
1,1,New York Liberty*,28.5,32,8,33,7,9.15,-1.09,8.06,...,14.2,25.1,.203,NaN,.476,14.5,79.2,.167,NaN,NaN
2,2,Connecticut Sun*,28.9,28,12,31,9,6.50,-0.75,5.75,...,13.8,25.2,.239,NaN,.482,17.6,78.2,.204,NaN,NaN
3,3,Minnesota Lynx*,27.9,30,10,30,10,6.38,-0.74,5.64,...,15.3,22.3,.182,NaN,.460,16.5,74.2,.181,NaN,NaN
4,4,Las Vegas Aces*,29.6,27,13,29,11,5.48,-0.70,4.77,...,12.4,16.6,.223,NaN,.488,14.1,79.5,.189,NaN,NaN


In [67]:
# flatten advanced-team header and update in place
for i, (name, df) in enumerate(team_tables):
    if name == "advanced-team":
        df.columns = df.iloc[0]  # use row 0 as header
        df = df[1:].reset_index(drop=True)  # drop that header row
        team_tables[i] = (name, df)  # update in list

In [68]:
for name, df in team_tables:
    if name == "advanced-team":
        print(name, df.shape)
        display(df.head())
        break

advanced-team (13, 29)


,Rk,Team,Age,W,L,PW,PL,MOV,SOS,SRS,...,TOV%,ORB%,FT/FGA,NaN,eFG%,TOV%,DRB%,FT/FGA,NaN,Arena
0,1,New York Liberty*,28.5,32,8,33,7,9.15,-1.09,8.06,...,14.2,25.1,.203,NaN,.476,14.5,79.2,.167,NaN,NaN
1,2,Connecticut Sun*,28.9,28,12,31,9,6.50,-0.75,5.75,...,13.8,25.2,.239,NaN,.482,17.6,78.2,.204,NaN,NaN
2,3,Minnesota Lynx*,27.9,30,10,30,10,6.38,-0.74,5.64,...,15.3,22.3,.182,NaN,.460,16.5,74.2,.181,NaN,NaN
3,4,Las Vegas Aces*,29.6,27,13,29,11,5.48,-0.70,4.77,...,12.4,16.6,.223,NaN,.488,14.1,79.5,.189,NaN,NaN
4,5,Seattle Storm*,29.1,25,15,27,13,4.48,-0.56,3.92,...,13.5,24.2,.211,NaN,.477,16.5,74.6,.210,NaN,NaN


In [69]:
# drop per_game_Rk from per_game-team table
team_tables = [(name, df.drop(columns=["Rk"]) if name == "advanced-team" and "Rk" in df.columns else df)
               for name, df in team_tables]

In [70]:
for name, df in team_tables:
    if name == "advanced-team":
        print(name, df.shape)
        display(df.head())
        break

advanced-team (13, 28)


,Team,Age,W,L,PW,PL,MOV,SOS,SRS,ORtg,...,TOV%,ORB%,FT/FGA,NaN,eFG%,TOV%,DRB%,FT/FGA,NaN,Arena
0,New York Liberty*,28.5,32,8,33,7,9.15,-1.09,8.06,109.6,...,14.2,25.1,.203,NaN,.476,14.5,79.2,.167,NaN,NaN
1,Connecticut Sun*,28.9,28,12,31,9,6.50,-0.75,5.75,105.0,...,13.8,25.2,.239,NaN,.482,17.6,78.2,.204,NaN,NaN
2,Minnesota Lynx*,27.9,30,10,30,10,6.38,-0.74,5.64,104.6,...,15.3,22.3,.182,NaN,.460,16.5,74.2,.181,NaN,NaN
3,Las Vegas Aces*,29.6,27,13,29,11,5.48,-0.70,4.77,108.0,...,12.4,16.6,.223,NaN,.488,14.1,79.5,.189,NaN,NaN
4,Seattle Storm*,29.1,25,15,27,13,4.48,-0.56,3.92,104.2,...,13.5,24.2,.211,NaN,.477,16.5,74.6,.210,NaN,NaN


In [71]:
# remove trailing '*' from all Team names
for i, (name, df) in enumerate(team_tables):
    if "Team" in df.columns:
        df["Team"] = df["Team"].str.replace("*", "", regex=False).str.strip()
        team_tables[i] = (name, df)

advanced-team (13, 28)


,Team,Age,W,L,PW,PL,MOV,SOS,SRS,ORtg,...,TOV%,ORB%,FT/FGA,NaN,eFG%,TOV%,DRB%,FT/FGA,NaN,Arena
0,New York Liberty,28.5,32,8,33,7,9.15,-1.09,8.06,109.6,...,14.2,25.1,.203,NaN,.476,14.5,79.2,.167,NaN,NaN
1,Connecticut Sun,28.9,28,12,31,9,6.50,-0.75,5.75,105.0,...,13.8,25.2,.239,NaN,.482,17.6,78.2,.204,NaN,NaN
2,Minnesota Lynx,27.9,30,10,30,10,6.38,-0.74,5.64,104.6,...,15.3,22.3,.182,NaN,.460,16.5,74.2,.181,NaN,NaN
3,Las Vegas Aces,29.6,27,13,29,11,5.48,-0.70,4.77,108.0,...,12.4,16.6,.223,NaN,.488,14.1,79.5,.189,NaN,NaN
4,Seattle Storm,29.1,25,15,27,13,4.48,-0.56,3.92,104.2,...,13.5,24.2,.211,NaN,.477,16.5,74.6,.210,NaN,NaN


In [76]:
# columns to drop from all tables except per_game-team
shared_cols = ["G", "MP"]

# drop G and MP from every table except per_game-team
for name, df in team_tables:
    if name != "per_game-team":
        df.drop(columns=[col for col in shared_cols if col in df.columns], inplace=True)

In [77]:
# separate into team and opponent tables
team_frames = [df for name, df in team_tables if name.endswith("-team")]
opp_frames = [df for name, df in team_tables if name.endswith("-opponent")]

# merge them
df_team = reduce(lambda left, right: pd.merge(left, right, on="Team", how="outer"), team_frames)
df_opp = reduce(lambda left, right: pd.merge(left, right, on="Team", how="outer"), opp_frames)

In [79]:
df_team.head()

,Team,G,MP,per_game_FG,per_game_FGA,per_game_FG%,per_game_3P,per_game_3PA,per_game_3P%,per_game_2P,...,shooting_FG% by Distance_3-10,shooting_FG% by Distance_10-16,shooting_FG% by Distance_16-3P,shooting_FG% by Distance_3P,shooting_Unnamed: 20_level_1,shooting_% of FG Ast'd_2P,shooting_% of FG Ast'd_3P,shooting_Unnamed: 23_level_1,shooting_Corner_%3PA,shooting_Corner_3P%
0,Atlanta Dream,40,201.9,27.8,68.1,0.408,6.0,19.4,0.308,21.8,...,0.374,0.339,0.395,0.308,NaN,0.599,0.891,NaN,0.210,0.288
1,Chicago Sky,40,200.0,29.7,70.3,0.422,4.8,14.9,0.323,24.9,...,0.377,0.358,0.344,0.324,NaN,0.606,0.808,NaN,0.195,0.310
2,Connecticut Sun,40,201.2,29.3,65.9,0.444,5.9,18.0,0.327,23.4,...,0.416,0.414,0.345,0.327,NaN,0.635,0.860,NaN,0.181,0.300
3,Dallas Wings,40,201.9,31.7,71.0,0.446,6.3,19.2,0.326,25.4,...,0.443,0.337,0.383,0.326,NaN,0.614,0.772,NaN,0.149,0.333
4,Indiana Fever,40,200.6,31.3,68.5,0.456,9.2,25.9,0.356,22.1,...,0.478,0.375,0.372,0.356,NaN,0.602,0.774,NaN,0.182,0.372


In [81]:
df.shape

(13, 23)

In [82]:
# show null counts for each column in df_team
df_team.isnull().sum().loc[lambda x: x > 0].sort_values(ascending=False)

,0
NaN,13
Arena,13
NaN,13
NaN,13
shooting_Unnamed: 23_level_1,13
shooting_Unnamed: 20_level_1,13
shooting_Unnamed: 13_level_1,13
shooting_Unnamed: 6_level_1,13
per_poss_FGA,1
per_poss_FG%,1


In [87]:
# drop unwanted columns from df_team
df_team.drop(columns=[
    "Arena",
    "NaN",
    "shooting_Unnamed: 23_level_1",
    "shooting_Unnamed: 20_level_1",
    "shooting_Unnamed: 13_level_1",
    "shooting_Unnamed: 6_level_1"
], inplace=True)

KeyError: "['Arena' 'NaN' 'shooting_Unnamed: 23_level_1'\n 'shooting_Unnamed: 20_level_1' 'shooting_Unnamed: 13_level_1'\n 'shooting_Unnamed: 6_level_1'] not found in axis"

In [90]:
# drop columns with name NaN
df_team.drop(columns=[col for col in df_team.columns if pd.isna(col)], inplace=True)

In [93]:
for col in df_team.columns:
    print(repr(col))

'Team'
'G'
'MP'
'per_game_FG'
'per_game_FGA'
'per_game_FG%'
'per_game_3P'
'per_game_3PA'
'per_game_3P%'
'per_game_2P'
'per_game_2PA'
'per_game_2P%'
'per_game_FT'
'per_game_FTA'
'per_game_FT%'
'per_game_ORB'
'per_game_DRB'
'per_game_TRB'
'per_game_AST'
'per_game_STL'
'per_game_BLK'
'per_game_TOV'
'per_game_PF'
'per_game_PTS'
'totals_FG'
'totals_FGA'
'totals_FG%'
'totals_3P'
'totals_3PA'
'totals_3P%'
'totals_2P'
'totals_2PA'
'totals_2P%'
'totals_FT'
'totals_FTA'
'totals_FT%'
'totals_ORB'
'totals_DRB'
'totals_TRB'
'totals_AST'
'totals_STL'
'totals_BLK'
'totals_TOV'
'totals_PF'
'totals_PTS'
'Age'
'W'
'L'
'PW'
'PL'
'MOV'
'SOS'
'SRS'
'ORtg'
'DRtg'
'NRtg'
'Pace'
'FTr'
'3PAr'
'TS%'
np.float64(nan)
'eFG%'
'TOV%'
'ORB%'
'FT/FGA'
np.float64(nan)
'eFG%'
'TOV%'
'DRB%'
'FT/FGA'
np.float64(nan)
'per_poss_FG'
'per_poss_FGA'
'per_poss_FG%'
'per_poss_3P'
'per_poss_3PA'
'per_poss_3P%'
'per_poss_2P'
'per_poss_2PA'
'per_poss_2P%'
'per_poss_FT'
'per_poss_FTA'
'per_poss_FT%'
'per_poss_ORB'
'per_poss_DRB'
'pe

In [105]:
df_team = df_team.loc[:, [not isinstance(col, float) or not np.isnan(col) for col in df_team.columns]]

In [106]:
# show null counts for each column in df_team
df_team.isnull().sum().loc[lambda x: x > 0].sort_values(ascending=False)

,0
W,1
L,1
NRtg,1
per_poss_FG,1
per_poss_FGA,1
per_poss_FG%,1
per_poss_3P,1
per_poss_3PA,1
per_poss_3P%,1
per_poss_2P,1


In [107]:
df_team[df_team.isnull().any(axis=1)]

,Team,G,MP,per_game_FG,per_game_FGA,per_game_FG%,per_game_3P,per_game_3PA,per_game_3P%,per_game_2P,...,shooting_FG% by Distance_2P,shooting_FG% by Distance_0-3,shooting_FG% by Distance_3-10,shooting_FG% by Distance_10-16,shooting_FG% by Distance_16-3P,shooting_FG% by Distance_3P,shooting_% of FG Ast'd_2P,shooting_% of FG Ast'd_3P,shooting_Corner_%3PA,shooting_Corner_3P%
6,League Average,40,201.0,29.9,68.3,0.438,7.7,22.8,0.338,22.2,...,0.488,0.654,0.432,0.386,0.379,0.338,0.62,0.876,0.19,0.356


In [108]:
df_team = df_team[df_team["Team"] != "League Average"]

In [109]:
# show null counts for each column in df_team
df_team.isnull().sum().loc[lambda x: x > 0].sort_values(ascending=False)

,0


In [110]:
for col in df_team.columns:
    print(repr(col))

'Team'
'G'
'MP'
'per_game_FG'
'per_game_FGA'
'per_game_FG%'
'per_game_3P'
'per_game_3PA'
'per_game_3P%'
'per_game_2P'
'per_game_2PA'
'per_game_2P%'
'per_game_FT'
'per_game_FTA'
'per_game_FT%'
'per_game_ORB'
'per_game_DRB'
'per_game_TRB'
'per_game_AST'
'per_game_STL'
'per_game_BLK'
'per_game_TOV'
'per_game_PF'
'per_game_PTS'
'totals_FG'
'totals_FGA'
'totals_FG%'
'totals_3P'
'totals_3PA'
'totals_3P%'
'totals_2P'
'totals_2PA'
'totals_2P%'
'totals_FT'
'totals_FTA'
'totals_FT%'
'totals_ORB'
'totals_DRB'
'totals_TRB'
'totals_AST'
'totals_STL'
'totals_BLK'
'totals_TOV'
'totals_PF'
'totals_PTS'
'Age'
'W'
'L'
'PW'
'PL'
'MOV'
'SOS'
'SRS'
'ORtg'
'DRtg'
'NRtg'
'Pace'
'FTr'
'3PAr'
'TS%'
'eFG%'
'TOV%'
'ORB%'
'FT/FGA'
'eFG%'
'TOV%'
'DRB%'
'FT/FGA'
'per_poss_FG'
'per_poss_FGA'
'per_poss_FG%'
'per_poss_3P'
'per_poss_3PA'
'per_poss_3P%'
'per_poss_2P'
'per_poss_2PA'
'per_poss_2P%'
'per_poss_FT'
'per_poss_FTA'
'per_poss_FT%'
'per_poss_ORB'
'per_poss_DRB'
'per_poss_TRB'
'per_poss_AST'
'per_poss_STL'
'per_p

In [111]:
df_opp.head()

,Team,per_game_FG,per_game_FGA,per_game_FG%,per_game_3P,per_game_3PA,per_game_3P%,per_game_2P,per_game_2PA,per_game_2P%,...,shooting_FG% by Distance_3-10,shooting_FG% by Distance_10-16,shooting_FG% by Distance_16-3P,shooting_FG% by Distance_3P,shooting_Unnamed: 20_level_1,shooting_% of FG Ast'd_2P,shooting_% of FG Ast'd_3P,shooting_Unnamed: 23_level_1,shooting_Corner_%3PA,shooting_Corner_3P%
0,Atlanta Dream,28.9,67.4,0.429,8.0,23.1,0.344,21.0,44.3,0.473,...,0.400,0.358,0.397,0.344,NaN,0.628,0.874,NaN,0.215,0.357
1,Chicago Sky,30.1,67.4,0.446,7.1,21.7,0.326,23.0,45.7,0.503,...,0.391,0.393,0.433,0.326,NaN,0.672,0.880,NaN,0.203,0.278
2,Connecticut Sun,27.2,63.0,0.431,6.5,20.6,0.313,20.7,42.4,0.488,...,0.435,0.401,0.362,0.313,NaN,0.644,0.895,NaN,0.193,0.346
3,Dallas Wings,33.5,70.5,0.475,8.6,23.4,0.365,24.9,47.1,0.530,...,0.468,0.400,0.397,0.365,NaN,0.620,0.898,NaN,0.197,0.357
4,Indiana Fever,31.2,70.6,0.441,9.2,25.5,0.361,21.9,45.1,0.487,...,0.448,0.375,0.408,0.361,NaN,0.603,0.889,NaN,0.193,0.396


In [112]:
df_opp.shape

(13, 86)

In [113]:
# show null counts for each column in df_team
df_opp.isnull().sum().loc[lambda x: x > 0].sort_values(ascending=False)

,0
shooting_Unnamed: 20_level_1,13
shooting_Unnamed: 13_level_1,13
shooting_Unnamed: 6_level_1,13
shooting_Unnamed: 23_level_1,13
per_game_FG%,1
...,...
per_poss_DRB,1
per_poss_PTS,1
per_poss_PF,1
per_poss_TOV,1


In [114]:
# columns to drop from df_opp
junk_cols_opp = [
    "shooting_Unnamed: 20_level_1",
    "shooting_Unnamed: 13_level_1",
    "shooting_Unnamed: 6_level_1",
    "shooting_Unnamed: 23_level_1"
]

# drop them if they exist
df_opp.drop(columns=[col for col in junk_cols_opp if col in df_opp.columns], inplace=True)

In [115]:
# show null counts for each column in df_team
df_opp.isnull().sum().loc[lambda x: x > 0].sort_values(ascending=False)

,0
per_game_FG,1
per_game_FGA,1
per_game_FG%,1
per_game_3P,1
per_game_3PA,1
...,...
per_poss_STL,1
per_poss_BLK,1
per_poss_TOV,1
per_poss_PF,1


In [116]:
df_opp[df_opp.isnull().any(axis=1)]

,Team,per_game_FG,per_game_FGA,per_game_FG%,per_game_3P,per_game_3PA,per_game_3P%,per_game_2P,per_game_2PA,per_game_2P%,...,shooting_FG% by Distance_2P,shooting_FG% by Distance_0-3,shooting_FG% by Distance_3-10,shooting_FG% by Distance_10-16,shooting_FG% by Distance_16-3P,shooting_FG% by Distance_3P,shooting_% of FG Ast'd_2P,shooting_% of FG Ast'd_3P,shooting_Corner_%3PA,shooting_Corner_3P%
6,League Average,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.488,0.654,0.432,0.386,0.379,0.338,0.62,0.876,0.19,0.356


In [117]:
df_opp = df_opp[df_opp["Team"] != "League Average"]

In [118]:
# show null counts for each column in df_team
df_opp.isnull().sum().loc[lambda x: x > 0].sort_values(ascending=False)

,0


In [120]:
for col in df_opp.columns:
    print(repr(col))

'Team'
'per_game_FG'
'per_game_FGA'
'per_game_FG%'
'per_game_3P'
'per_game_3PA'
'per_game_3P%'
'per_game_2P'
'per_game_2PA'
'per_game_2P%'
'per_game_FT'
'per_game_FTA'
'per_game_FT%'
'per_game_ORB'
'per_game_DRB'
'per_game_TRB'
'per_game_AST'
'per_game_STL'
'per_game_BLK'
'per_game_TOV'
'per_game_PF'
'per_game_PTS'
'totals_FG'
'totals_FGA'
'totals_FG%'
'totals_3P'
'totals_3PA'
'totals_3P%'
'totals_2P'
'totals_2PA'
'totals_2P%'
'totals_FT'
'totals_FTA'
'totals_FT%'
'totals_ORB'
'totals_DRB'
'totals_TRB'
'totals_AST'
'totals_STL'
'totals_BLK'
'totals_TOV'
'totals_PF'
'totals_PTS'
'per_poss_FG'
'per_poss_FGA'
'per_poss_FG%'
'per_poss_3P'
'per_poss_3PA'
'per_poss_3P%'
'per_poss_2P'
'per_poss_2PA'
'per_poss_2P%'
'per_poss_FT'
'per_poss_FTA'
'per_poss_FT%'
'per_poss_ORB'
'per_poss_DRB'
'per_poss_TRB'
'per_poss_AST'
'per_poss_STL'
'per_poss_BLK'
'per_poss_TOV'
'per_poss_PF'
'per_poss_PTS'
'shooting_FG%'
'shooting_Dist.'
'shooting_% of FGA by Distance_2P'
'shooting_% of FGA by Distance_0-3'
's

In [121]:
# rename df_team columns
df_team.columns = [
    col if col in ["Team", "G", "MP"] else f"team_{col}"
    for col in df_team.columns
]

# rename df_opp columns
df_opp.columns = [
    col if col == "Team" else f"opp_{col}"
    for col in df_opp.columns
]

In [122]:
df_team.head()

,Team,G,MP,team_per_game_FG,team_per_game_FGA,team_per_game_FG%,team_per_game_3P,team_per_game_3PA,team_per_game_3P%,team_per_game_2P,...,team_shooting_FG% by Distance_2P,team_shooting_FG% by Distance_0-3,team_shooting_FG% by Distance_3-10,team_shooting_FG% by Distance_10-16,team_shooting_FG% by Distance_16-3P,team_shooting_FG% by Distance_3P,team_shooting_% of FG Ast'd_2P,team_shooting_% of FG Ast'd_3P,team_shooting_Corner_%3PA,team_shooting_Corner_3P%
0,Atlanta Dream,40,201.9,27.8,68.1,0.408,6.0,19.4,0.308,21.8,...,0.448,0.616,0.374,0.339,0.395,0.308,0.599,0.891,0.210,0.288
1,Chicago Sky,40,200.0,29.7,70.3,0.422,4.8,14.9,0.323,24.9,...,0.449,0.587,0.377,0.358,0.344,0.324,0.606,0.808,0.195,0.310
2,Connecticut Sun,40,201.2,29.3,65.9,0.444,5.9,18.0,0.327,23.4,...,0.487,0.636,0.416,0.414,0.345,0.327,0.635,0.860,0.181,0.300
3,Dallas Wings,40,201.9,31.7,71.0,0.446,6.3,19.2,0.326,25.4,...,0.490,0.668,0.443,0.337,0.383,0.326,0.614,0.772,0.149,0.333
4,Indiana Fever,40,200.6,31.3,68.5,0.456,9.2,25.9,0.356,22.1,...,0.517,0.622,0.478,0.375,0.372,0.356,0.602,0.774,0.182,0.372


In [123]:
df_opp.head()

,Team,opp_per_game_FG,opp_per_game_FGA,opp_per_game_FG%,opp_per_game_3P,opp_per_game_3PA,opp_per_game_3P%,opp_per_game_2P,opp_per_game_2PA,opp_per_game_2P%,...,opp_shooting_FG% by Distance_2P,opp_shooting_FG% by Distance_0-3,opp_shooting_FG% by Distance_3-10,opp_shooting_FG% by Distance_10-16,opp_shooting_FG% by Distance_16-3P,opp_shooting_FG% by Distance_3P,opp_shooting_% of FG Ast'd_2P,opp_shooting_% of FG Ast'd_3P,opp_shooting_Corner_%3PA,opp_shooting_Corner_3P%
0,Atlanta Dream,28.9,67.4,0.429,8.0,23.1,0.344,21.0,44.3,0.473,...,0.473,0.617,0.400,0.358,0.397,0.344,0.628,0.874,0.215,0.357
1,Chicago Sky,30.1,67.4,0.446,7.1,21.7,0.326,23.0,45.7,0.503,...,0.503,0.650,0.391,0.393,0.433,0.326,0.672,0.880,0.203,0.278
2,Connecticut Sun,27.2,63.0,0.431,6.5,20.6,0.313,20.7,42.4,0.488,...,0.488,0.660,0.435,0.401,0.362,0.313,0.644,0.895,0.193,0.346
3,Dallas Wings,33.5,70.5,0.475,8.6,23.4,0.365,24.9,47.1,0.530,...,0.530,0.705,0.468,0.400,0.397,0.365,0.620,0.898,0.197,0.357
4,Indiana Fever,31.2,70.6,0.441,9.2,25.5,0.361,21.9,45.1,0.487,...,0.487,0.608,0.448,0.375,0.408,0.361,0.603,0.889,0.193,0.396


In [124]:
df = pd.merge(df_team, df_opp, on="Team", how="outer")

In [125]:
df.head()

,Team,G,MP,team_per_game_FG,team_per_game_FGA,team_per_game_FG%,team_per_game_3P,team_per_game_3PA,team_per_game_3P%,team_per_game_2P,...,opp_shooting_FG% by Distance_2P,opp_shooting_FG% by Distance_0-3,opp_shooting_FG% by Distance_3-10,opp_shooting_FG% by Distance_10-16,opp_shooting_FG% by Distance_16-3P,opp_shooting_FG% by Distance_3P,opp_shooting_% of FG Ast'd_2P,opp_shooting_% of FG Ast'd_3P,opp_shooting_Corner_%3PA,opp_shooting_Corner_3P%
0,Atlanta Dream,40,201.9,27.8,68.1,0.408,6.0,19.4,0.308,21.8,...,0.473,0.617,0.400,0.358,0.397,0.344,0.628,0.874,0.215,0.357
1,Chicago Sky,40,200.0,29.7,70.3,0.422,4.8,14.9,0.323,24.9,...,0.503,0.650,0.391,0.393,0.433,0.326,0.672,0.880,0.203,0.278
2,Connecticut Sun,40,201.2,29.3,65.9,0.444,5.9,18.0,0.327,23.4,...,0.488,0.660,0.435,0.401,0.362,0.313,0.644,0.895,0.193,0.346
3,Dallas Wings,40,201.9,31.7,71.0,0.446,6.3,19.2,0.326,25.4,...,0.530,0.705,0.468,0.400,0.397,0.365,0.620,0.898,0.197,0.357
4,Indiana Fever,40,200.6,31.3,68.5,0.456,9.2,25.9,0.356,22.1,...,0.487,0.608,0.448,0.375,0.408,0.361,0.603,0.889,0.193,0.396


In [126]:
df.shape

(12, 188)

In [127]:
# save to CSV
df.to_csv("2024_team_data.csv", index=False)

# download to local machine
from google.colab import files
files.download("2024_team_data.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>